# A. Cargar Datos

In [1]:
import pandas as pd
from plotly import express as px
import streamlit as st
import plotly.io as pio
import numpy as np
pio.renderers.default = "notebook_connected"

In [2]:
# Cargar conjunto de datos, como pandas.DataFrame
df = pd.read_csv('vehicles_us.csv')

# 1. Exploracion

## 1.1 Inspeccion inicial

In [3]:
# Exploracion de datos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB


In [4]:
df.sample(random_state=25000, n=15)

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
24014,2400,2013.0,chevrolet malibu,salvage,NaN,gas,NaN,automatic,sedan,silver,NaN,2018-05-18,51
47815,35985,2016.0,ram 3500,like new,6.0,diesel,53191.0,automatic,truck,grey,1.0,2018-09-05,31
28710,28899,2013.0,jeep wrangler unlimited,like new,NaN,gas,34000.0,automatic,SUV,white,1.0,2018-09-26,32
40760,4000,2000.0,chevrolet silverado,excellent,8.0,gas,120000.0,automatic,truck,white,NaN,2019-02-23,22
9963,26490,2013.0,chevrolet silverado 3500hd,excellent,8.0,gas,NaN,automatic,other,white,NaN,2019-01-15,49
16195,800,1998.0,jeep grand cherokee laredo,fair,6.0,gas,220500.0,automatic,SUV,grey,1.0,2018-05-26,69
4734,28977,2012.0,ram 2500,excellent,NaN,gas,87742.0,automatic,pickup,NaN,1.0,2019-04-10,71
50498,10500,2017.0,ford focus,excellent,4.0,gas,40000.0,automatic,sedan,NaN,NaN,2019-01-03,37
13176,4170,2004.0,ford ranger,good,4.0,gas,115445.0,other,truck,grey,NaN,2019-03-03,8
41204,8995,NaN,ford f-150,good,8.0,gas,131000.0,automatic,pickup,red,1.0,2019-02-18,17


In [5]:
# Generar estadisticos descriptivos, por tipo de datos (numericos, categoricos)
print('=== DESCRIPCION DE COLUMNAS NUMERICAS ===\n')
print(df.describe(include='number').round(2))

print('\n=== DESCRIPCION DE COLUMNAS CATEGORICAS === \n')
print(df.describe(include=['object','str']))

=== DESCRIPCION DE COLUMNAS NUMERICAS ===

           price  model_year  cylinders   odometer   is_4wd  days_listed
count   51525.00    47906.00   46265.00   43633.00  25572.0     51525.00
mean    12132.46     2009.75       6.13  115553.46      1.0        39.55
std     10040.80        6.28       1.66   65094.61      0.0        28.20
min         1.00     1908.00       3.00       0.00      1.0         0.00
25%      5000.00     2006.00       4.00   70000.00      1.0        19.00
50%      9000.00     2011.00       6.00  113000.00      1.0        33.00
75%     16839.00     2014.00       8.00  155000.00      1.0        53.00
max    375000.00     2019.00      12.00  990000.00      1.0       271.00

=== DESCRIPCION DE COLUMNAS CATEGORICAS === 

             model  condition   fuel transmission   type paint_color  \
count        51525      51525  51525        51525  51525       42258   
unique         100          6      5            3     13          12   
top     ford f-150  excellent    gas 

`1.` En la columna 'price' observamos que se tiene un valor minimo de 1.00, lo cual se atribuye a tacticas de los vendedores (el precio es a tratar, se coloca un valor "dummy" para cumplir con el requerimiento)

`2.` La columna 'is_4wd' parece ser un booleano, la uniformidad de los datos (std=0) sugiere que hay un problema: solo se tienen datos de vehiculos para los cuales el valor es 1.0 (o True). En los demas casos, no existen datos por lo cual esta caracteristica permanece ambigua. Por el momento, no se tomara accion al respecto.

## 1.2 Conversion de dtypes

In [6]:
df['model_year'] = pd.to_numeric(
    df['model_year'], errors='coerce'
).astype('Int64')

df['cylinders'] = pd.to_numeric(
    df['cylinders'], errors='coerce'
).astype('Int64')

df['odometer'] = pd.to_numeric(
    df['odometer'], errors='coerce'
).astype('Int64')

`3.` Las columnas 'model_year', 'cylinders' y 'odometer' se convierten a dtype Int64 (numero entero anulable) pues refleja de forma mas precisa la naturaleza de estas variables, ademas facilitara los analisis y agrupamientos.

In [7]:
df['date_posted'] = pd.to_datetime(df['date_posted'], format='%Y-%m-%d')

`4.` El dtype de la columna 'date_posted' se cambian de tipo 'str' a tipo 'datetime' para que haya congruencia entre los datos y su tipo, ademas de facilitar los analisis posteriores.

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   price         51525 non-null  int64         
 1   model_year    47906 non-null  Int64         
 2   model         51525 non-null  str           
 3   condition     51525 non-null  str           
 4   cylinders     46265 non-null  Int64         
 5   fuel          51525 non-null  str           
 6   odometer      43633 non-null  Int64         
 7   transmission  51525 non-null  str           
 8   type          51525 non-null  str           
 9   paint_color   42258 non-null  str           
 10  is_4wd        25572 non-null  float64       
 11  date_posted   51525 non-null  datetime64[us]
 12  days_listed   51525 non-null  int64         
dtypes: Int64(3), datetime64[us](1), float64(1), int64(2), str(6)
memory usage: 7.3 MB


# 2. Preparacion

## 2.1 Datos duplicados

In [9]:
print(df.duplicated().sum())

0


`5.` No se encuentran duplicados explicitos.

In [10]:
df['condition'].value_counts()

condition
excellent    24773
good         20145
like new      4742
fair          1607
new            143
salvage        115
Name: count, dtype: int64

In [11]:
df['fuel'].value_counts()

fuel
gas         47288
diesel       3714
hybrid        409
other         108
electric        6
Name: count, dtype: int64

In [12]:
df['transmission'].value_counts()

transmission
automatic    46902
manual        2829
other         1794
Name: count, dtype: int64

`6.` Se evalúan los valores categóricos en columnas 'condition', 'fuel' y 'transmission' sin detectarse duplicados implícitos.

## 2.2 Datos ausentes

In [13]:
# Obtener conteo de valores ausentes, pasar resultado a dataframe
miss_count = (df.isna().sum().to_frame(name='missing_count'))

# Complementar dataframe con campo calculado: % del total
miss_count['% of total'] = (
    100 * miss_count['missing_count'] / len(df)
).round(2)

miss_count

,missing_count,% of total
price,0,0.00
model_year,3619,7.02
model,0,0.00
condition,0,0.00
cylinders,5260,10.21
fuel,0,0.00
odometer,7892,15.32
transmission,0,0.00
type,0,0.00
paint_color,9267,17.99


`7.` Se observa una cantidad importante de registros ausentes en columna 'is_4wd'. Esta columna indica si un vehículo cuenta con tracción en las 4 ruedas (four-wheel drive, como se conoce en inglés). Se tiene aproximadamente la mitad de registros ausentes. Esta columna se deja sin cambios, ya que la proporción de datos ausentes no permite considerar ni la imputación de datos ni valores sentinela (utilidad negligible).

In [14]:
df[df['model_year'].isna()].sample(15)

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
45230,13995,<NA>,chevrolet silverado 1500,excellent,8,gas,151442,automatic,truck,NaN,1.0,2018-09-06,33
2627,7356,<NA>,chevrolet impala,excellent,6,gas,113366,automatic,sedan,NaN,NaN,2018-12-13,3
30022,5000,<NA>,chevrolet impala,good,6,gas,<NA>,automatic,sedan,black,NaN,2018-07-19,38
25483,6650,<NA>,ford explorer,good,6,gas,142000,automatic,SUV,white,NaN,2018-07-27,21
23649,6599,<NA>,ram 1500,like new,8,gas,<NA>,automatic,pickup,NaN,NaN,2019-03-30,28
32905,4995,<NA>,volkswagen passat,good,5,gas,114763,manual,sedan,white,NaN,2019-01-01,35
17277,6573,<NA>,chrysler 300,excellent,6,gas,141016,automatic,sedan,red,NaN,2018-11-04,10
8375,18900,<NA>,toyota tacoma,good,6,gas,94560,automatic,truck,grey,1.0,2019-03-03,117
12562,13774,<NA>,chevrolet malibu,good,4,gas,53212,automatic,sedan,white,NaN,2018-06-22,27
30305,7277,<NA>,ford f150,excellent,8,gas,135000,automatic,truck,brown,NaN,2018-06-20,93


## 2.3 Enriquecimiento de datos

In [15]:
words = df['model'].str.split(" ")
df['brand'] = words.str[0]

In [16]:
df['brand'].value_counts()

brand
ford             12672
chevrolet        10611
toyota            5445
honda             3485
ram               3316
jeep              3281
nissan            3208
gmc               2378
subaru            1272
dodge             1255
hyundai           1173
volkswagen         869
chrysler           838
kia                585
cadillac           322
buick              271
bmw                267
acura              236
mercedes-benz       41
Name: count, dtype: int64

In [17]:
df['log_price'] = np.log10(df['price']+1)
df['log_odometer'] = np.log10(df['odometer']+1)

In [40]:
df['model_year'].describe()

count       47906.0
mean     2009.75047
std        6.282065
min          1908.0
25%          2006.0
50%          2011.0
75%          2014.0
max          2019.0
Name: model_year, dtype: Float64

In [42]:
df['age'] = 2020 - df['model_year']

# 3. Análisis

In [44]:
# Get the top 10 brands as a list
top_10_brands = (df['brand']
    .value_counts()
    .nlargest(10)
    .index.tolist()
                )

# Filter the dataframe
df_top10 = df[df['brand'].isin(top_10_brands)]
df_top10 = df_top10.query('price > 1')

df_top10['odometer'] = df_top10['odometer'].dropna()
df_top10['model_year'] = df_top10['model_year'].dropna()

In [46]:
df_top10[['log_price', 'model_year', 'odometer', 'age']].corr().round(3)

,log_price,model_year,odometer,age
log_price,1.000,0.508,-0.463,-0.508
model_year,0.508,1.000,-0.460,-1.000
odometer,-0.463,-0.460,1.000,0.460
age,-0.508,-1.000,0.460,1.000


## 3.1 Gráfico de dispersión

In [54]:
fig_1 = px.scatter(
    df_top10,
    x="age",
    y="log_price",
)

fig_1.show()

In [38]:
fig_2 = px.scatter(
    df_top10,
    x="odometer",
    y="log_price",
)

fig_2.show()

In [55]:
condition_order = ['salvage', 'fair', 'good', 'excellent', 'like new', 'new']

df_top10['condition'] = pd.Categorical(
    df_top10['condition'],
    categories=condition_order,
    ordered=True
)

fig_3 = px.scatter(
    df_top10,
    x="condition",
    y="log_price",
    category_orders={"condition": condition_order}
)

fig_3.show()

## 3.2 Histograma

In [62]:
fig4 = px.histogram(df_top10, x='price', nbins=80, color='condition')

fig4.show()